# Doughnut of Social and Planetary Boundaries

This notebook reproduces the summary tables for the social foundation (Table 1) and ecological ceiling (Table 2) from [Fanning & Raworth (2025)](https://www.nature.com/articles/s41586-025-09385-1), who examined progress toward meeting the needs of all people within the means of the living planet.

Data source: [Zenodo repository](https://doi.org/10.5281/zenodo.15688961), covering 35 indicators over the 2000–2022 period.

In [ ]:
import pandas as pd
import numpy as np
import glob
import os

BASE = os.path.join('a-fanning', 'a-fanning-doughnut-v3-a0460e5', 'Analysis-Final')


def style_summary_table(df):
    """Style an indicator summary DataFrame with centered headers and left-aligned text columns."""
    return (
        df.style
        .set_table_styles([
            {'selector': 'th.col_heading', 'props': 'text-align: center;'},
            {'selector': 'th.col_heading.level0', 'props': 'text-align: center;'},
            {'selector': 'th.col_heading.level1', 'props': 'text-align: center;'},
        ])
        .set_properties(
            subset=[('dimension', ''), ('indicator', '')],
            **{'text-align': 'left'},
        )
        .hide(axis='index')
        .format(
            {('value', 'first'): '{:.2f}', ('value', 'last'): '{:.2f}'},
        )
    )

## Part 1: The Social Foundation — Table 1

The social foundation defines the minimum level of human well-being below which critical deprivation exists. Fanning & Raworth track **21 indicators** across **12 dimensions** (food, health, education, energy, etc.), each measuring the share of the global population experiencing a specific form of deprivation.

Several raw indicators record *positive* outcomes (e.g., literacy rate, energy access) and are inverted (`100 − value`) to express the corresponding deprivation rate. Two indicators on a 0–1 scale (Gender Inequality Index and societal poverty) are rescaled to percentages.

The table below shows the **first and last available global values** for each indicator over the 2000–2022 period.

In [ ]:
soc_files = sorted(glob.glob(os.path.join(BASE, 'cleanData', 'soc-*_clean.csv')))
soc_all = pd.concat([pd.read_csv(f) for f in soc_files], ignore_index=True)

# Keep only global ("World") observations with valid values
soc_world = soc_all[(soc_all['group'] == 'World') & soc_all['value'].notna()].copy()

# Map indicator codes to human-readable deprivation descriptions
SOCIAL_NAMES = {
    'internet': 'Population not accessing the internet',
    'publicTrans': (
        'Urban population lacking convenient access to public<br>transport'
    ),
    'adultLiteracy': 'Adult population (aged 15+ years) who are illiterate',
    'secondarySchool': (
        'Young adult population (aged 21-23 years) with<br>'
        'incomplete upper secondary education'
    ),
    'energyAccess': 'Population lacking access to electricity',
    'energyIndoor': (
        'Population lacking access to clean fuels and<br>'
        'technologies for cooking, heating and lighting'
    ),
    'genderGapIndex': (
        'Population-weighted score on the Gender Inequality<br>'
        'Index (global gap between women and men in terms of<br>'
        'reproductive health, empowerment and employment)'
    ),
    'foodInsecurity': 'Population with moderate to severe food insecurity',
    'undernourishment': 'Population undernourished',
    'UHCindex': (
        'Population living in countries without high coverage<br>'
        'of essential health services (Universal Health<br>'
        'Coverage Index score less than 60 out of 100)'
    ),
    'under5death': (
        'Population living in countries with under-5<br>'
        'mortality rate exceeding 25 per 1,000 live births'
    ),
    'urbanSlums': 'Urban population living in slums or informal settlements',
    'societalPoverty': (
        'Population living below the societal poverty line,<br>'
        " set at half their country's median household<br>"
        'income or at least $15 a day'
    ),
    'youthNEET': (
        'Population of young people (aged 15-24 years) not<br>'
        'in employment, education or training'
    ),
    'controlCorruption': (
        'Population stating that they perceive<br>'
        'widespread corruption in government and business'
    ),
    'homicideOver5': (
        'Population living in countries with a homicide rate<br>'
        'of 5 or more per 100,000'
    ),
    'govRegimes': 'Population living in countries governed by an autocratic regime',
    'palma': (
        'Population living in countries with a Palma ratio of 2 or more<br>'
        ' (the income share of the richest 10% of people relative<br>'
        'to the poorest 40%)'
    ),
    'socialSupport': (
        'Population stating that they are without someone to<br>'
        'count on in times of trouble'
    ),
    'drinkingH2O': 'Population lacking access to safely managed drinking water',
    'sanitation': 'Population lacking access to safely managed sanitation',
}

soc_world = soc_world[soc_world['indicator'].isin(SOCIAL_NAMES)].copy()

# Indicators that record positive outcomes need inversion to express deprivation
INVERT = {'internet', 'publicTrans', 'adultLiteracy', 'secondarySchool',
          'energyAccess', 'energyIndoor', 'drinkingH2O', 'sanitation'}
# Indicators on a 0-1 scale need rescaling to percentages
SCALE100 = {'genderGapIndex', 'societalPoverty'}

mask_inv = soc_world['indicator'].isin(INVERT)
soc_world.loc[mask_inv, 'value'] = 100 - soc_world.loc[mask_inv, 'value']

mask_sc = soc_world['indicator'].isin(SCALE100)
soc_world.loc[mask_sc, 'value'] = soc_world.loc[mask_sc, 'value'] * 100

# Display order: alphabetical by dimension, then by indicator within each dimension
SOC_ORDER = [
    ('connectivity', 'internet'), ('connectivity', 'publicTrans'),
    ('education', 'adultLiteracy'), ('education', 'secondarySchool'),
    ('energy', 'energyAccess'), ('energy', 'energyIndoor'),
    ('equality', 'genderGapIndex'),
    ('food', 'foodInsecurity'), ('food', 'undernourishment'),
    ('health', 'UHCindex'), ('health', 'under5death'),
    ('housing', 'urbanSlums'),
    ('income and work', 'societalPoverty'), ('income and work', 'youthNEET'),
    ('peace and justice', 'controlCorruption'), ('peace and justice', 'homicideOver5'),
    ('political voice', 'govRegimes'),
    ('social cohesion', 'palma'), ('social cohesion', 'socialSupport'),
    ('water', 'drinkingH2O'), ('water', 'sanitation'),
]

rows = []
for dim, ind in SOC_ORDER:
    grp = soc_world[(soc_world['dimension'] == dim) & (soc_world['indicator'] == ind)]
    grp = grp.sort_values('date')
    if grp.empty:
        continue
    first, last = grp.iloc[0], grp.iloc[-1]
    rows.append([
        dim.capitalize(), SOCIAL_NAMES[ind],
        int(first['date']), int(last['date']),
        first['value'], last['value'],
    ])

columns = pd.MultiIndex.from_arrays([
    ['dimension', 'indicator', 'date', 'date', 'value', 'value'],
    ['', '', 'first', 'last', 'first', 'last'],
])
table1 = pd.DataFrame(rows, columns=columns)

style_summary_table(table1)

## Part 2: The Ecological Ceiling — Table 2

The ecological ceiling defines the maximum level of resource use and environmental impact that the Earth system can sustain. Fanning & Raworth track **13 indicators** across **9 dimensions**, each compared against a planetary boundary threshold. Exceeding the boundary means the indicator is in *overshoot*.

Unlike the social indicators, ecological values are reported in their original physical units (ppm, Mt, etc.) and do not require transformation. The boundary threshold is noted in each indicator description.

In [ ]:
eco_files = sorted(glob.glob(os.path.join(BASE, 'cleanData', 'eco-*_clean.csv')))
eco_all = pd.concat([pd.read_csv(f) for f in eco_files], ignore_index=True)

# Keep only global doughnut observations with valid values
eco_world = eco_all[
    (eco_all['group'] == 'World')
    & (eco_all['type'] == 'global doughnut')
    & eco_all['value'].notna()
].copy()

# Map indicator codes to descriptions including boundary thresholds
ECO_NAMES = {
    'interhemAOD': (
        'Asymmetry between Earth\'s hemispheres of sunlight<br>'
        'reaching the surface, owing to differences in atmospheric particle<br>'
        'concentration (at most 0.1 inter-hemispheric difference in Aerosol<br>'
        'Optical Depth)'
    ),
    'extinction1900': (
        'Rate of species extinctions per million species<br>'
        'years (at most 10 E/MSY)'
    ),
    'hanppGtC': (
        'Human appropriation of net primary productivity,<br>'
        'billions of tonnes of carbon per year (at most 10% of 55.9 Gt C)'
    ),
    'chemicalsMt': (
        'Production of hazardous chemicals, millions of tonnes<br>'
        'per year (at most 5% of the 1,200 Mt of total chemicals<br>'
        'produced in year 2000)'
    ),
    'co2_ppm': (
        'Atmospheric carbon dioxide concentration, parts per million<br>'
        '(at most 350 ppm CO2)'
    ),
    'erf_wm2': (
        'Human-induced radiative forcing at the top of the atmosphere,<br>'
        ' Watt per square metre (at most 1 W m**(-2))'
    ),
    'blueDev': (
        'Proportion of land area with human-induced disturbance of blue-water<br>'
        'flow deviating from Holocene variability (at most 10.2%)'
    ),
    'soilDev': (
        'Proportion of land area with root-zone soil moisture deviating from<br>'
        'Holocene variability (at most 11.1%)'
    ),
    'forestAreaMKM2': (
        'Area of forested land as a proportion of forest-covered land before<br>'
        'human alteration (at least 75% of 64 million square kilometres)'
    ),
    'nitrogenMt': (
        'Nitrogen applied to land as fertilizer, millions of tonnes per year<br>'
        '(at most 62 Mt per year)'
    ),
    'phosphorusMt': (
        'Phosphorus applied to land as fertilizer, millions of tonnes per year<br>'
        '(at most 6.2 Mt per year)'
    ),
    'omega_a': (
        'Average saturation state of aragonite at the ocean surface<br>'
        '(at least 80% of pre-industrial saturation state of 3.44 \u03a9arag)'
    ),
    'totalOzone': (
        'Concentration of ozone in the stratosphere, Dobson units<br>'
        '(at most 5% decrease with respect to 1964-1980 value of 290 DU)'
    ),
}

eco_world = eco_world[eco_world['indicator'].isin(ECO_NAMES)].copy()

# Display order: alphabetical by dimension, then by indicator within each dimension
ECO_ORDER = [
    ('air pollution', 'interhemAOD'),
    ('biodiversity breakdown', 'extinction1900'),
    ('biodiversity breakdown', 'hanppGtC'),
    ('chemical pollution', 'chemicalsMt'),
    ('climate change', 'co2_ppm'),
    ('climate change', 'erf_wm2'),
    ('freshwater disruption', 'blueDev'),
    ('freshwater disruption', 'soilDev'),
    ('land conversion', 'forestAreaMKM2'),
    ('nutrient pollution', 'nitrogenMt'),
    ('nutrient pollution', 'phosphorusMt'),
    ('ocean acidification', 'omega_a'),
    ('ozone depletion', 'totalOzone'),
]

rows = []
for dim, ind in ECO_ORDER:
    grp = eco_world[(eco_world['dimension'] == dim) & (eco_world['indicator'] == ind)]
    grp = grp.sort_values('date')
    if grp.empty:
        continue
    first, last = grp.iloc[0], grp.iloc[-1]
    rows.append([
        dim.capitalize(), ECO_NAMES[ind],
        int(first['date']), int(last['date']),
        first['value'], last['value'],
    ])

columns = pd.MultiIndex.from_arrays([
    ['dimension', 'indicator', 'date', 'date', 'value', 'value'],
    ['', '', 'first', 'last', 'first', 'last'],
])
table2 = pd.DataFrame(rows, columns=columns)

style_summary_table(table2)